# Load power spectra

In [2]:
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp

# Load our simulation results at z=1.650, mean of the two inverse-seed runs
k_LCDM_Me_a, Pk_LCDM_Me_a = load_power_spectrum(hp.pofk_cb_file('COLA_NM4096', 'LCDM', 's1234-fixed', float(redshift_label)))
k_LCDM_Me_ma, Pk_LCDM_Me_ma = load_power_spectrum(hp.pofk_cb_file('COLA_NM4096', 'LCDM', 's1234-fixed-inv', float(redshift_label)))
k_nDGP_Me_a, Pk_nDGP_Me_a = load_power_spectrum(hp.pofk_cb_file('COLA_NM4096', 'nDGP_N1.2', 's1234-fixed', float(redshift_label)))
k_nDGP_Me_ma, Pk_nDGP_Me_ma = load_power_spectrum(hp.pofk_cb_file('COLA_NM4096', 'nDGP_N1.2', 's1234-fixed-inv', float(redshift_label)))

k_LCDM_Me_inv = np.asarray(k_LCDM_Me_a)
Pk_LCDM_Me_inv = np.vstack([Pk_LCDM_Me_a, Pk_LCDM_Me_ma]).mean(axis=0)
k_nDGP_Me_inv = np.asarray(k_nDGP_Me_a)
Pk_nDGP_Me_inv = np.vstack([Pk_nDGP_Me_a, Pk_nDGP_Me_ma]).mean(axis=0)

NameError: name 'load_power_spectrum' is not defined

# Plot 1

In [ ]:
# Four-panel comparison at z=1.650 (inverse-seed mean)
k_LCDM_Bart_plot = np.asarray(k_LCDM_Bart)
Pk_LCDM_Bart_plot = np.asarray(Pk_LCDM_Bart)
Pk_LCDM_Bart_std_plot = np.asarray(Pk_LCDM_Bart_std)
k_nDGP_Bart_plot = np.asarray(k_nDGP_Bart)
Pk_nDGP_Bart_plot = np.asarray(Pk_nDGP_Bart)
bart_label = r'$\mathrm{COLA\_NM4096\ mean}$'

k_LCDM_Me_plot = np.asarray(k_LCDM_Me_inv)
Pk_LCDM_Me_plot = np.asarray(Pk_LCDM_Me_inv)
k_nDGP_Me_plot = np.asarray(k_nDGP_Me_inv)
Pk_nDGP_Me_plot = np.asarray(Pk_nDGP_Me_inv)

# Cut our power spectra to the comparison's k range
mask_LCDM_Me = (k_LCDM_Me_plot >= k_LCDM_Bart_plot.min()) & (k_LCDM_Me_plot <= k_LCDM_Bart_plot.max())
k_LCDM_Me_plot = k_LCDM_Me_plot[mask_LCDM_Me]
Pk_LCDM_Me_plot = Pk_LCDM_Me_plot[mask_LCDM_Me]
mask_nDGP_Me = (k_nDGP_Me_plot >= k_nDGP_Bart_plot.min()) & (k_nDGP_Me_plot <= k_nDGP_Bart_plot.max())
k_nDGP_Me_plot = k_nDGP_Me_plot[mask_nDGP_Me]
Pk_nDGP_Me_plot = Pk_nDGP_Me_plot[mask_nDGP_Me]

Pk_nDGP_Me_on_LCDM = np.interp(k_LCDM_Me_plot, k_nDGP_Me_plot, Pk_nDGP_Me_plot)
Pk_LCDM_Me_on_Bart = np.interp(k_LCDM_Bart_plot, k_LCDM_Me_plot, Pk_LCDM_Me_plot)

Pk_LCDM_Bart_low = np.maximum(Pk_LCDM_Bart_plot - Pk_LCDM_Bart_std_plot, 1e-30)
Pk_LCDM_Bart_high = Pk_LCDM_Bart_plot + Pk_LCDM_Bart_std_plot
Pk_COLA_over_Bart = Pk_LCDM_Me_on_Bart / Pk_LCDM_Bart_plot
Pk_COLA_over_Bart_low = Pk_LCDM_Me_on_Bart / Pk_LCDM_Bart_high
Pk_COLA_over_Bart_high = Pk_LCDM_Me_on_Bart / Pk_LCDM_Bart_low

fig_size = (6.6, 4.35)
label_size = 10.5
tick_size = 8.8
legend_size = 8.3
line_width = 1.45
std_alpha = 0.18
cola_color = '#c47f2b'
bart_color = '#1f4e79'
ratio_color = '#2b7a78'
band_color = bart_color
ref_color = '0.35'

fig, axes = plt.subplots(2, 2, figsize=fig_size, dpi=300, constrained_layout=True)
ax_tl, ax_tr = axes[0]
ax_bl, ax_br = axes[1]

ax_tl.semilogx(k_LCDM_Me_plot, k_LCDM_Me_plot**(3/2) * Pk_LCDM_Me_plot, lw=line_width, color=cola_color, ls='--', label=r'$\mathrm{Our\ simulation\ (mean,\ inverse\ seeds)}$')
ax_tl.semilogx(k_LCDM_Bart_plot, k_LCDM_Bart_plot**(3/2) * Pk_LCDM_Bart_plot, lw=line_width, color=bart_color, label=bart_label)
ax_tl.fill_between(
    k_LCDM_Bart_plot,
    k_LCDM_Bart_plot**(3/2) * Pk_LCDM_Bart_low,
    k_LCDM_Bart_plot**(3/2) * Pk_LCDM_Bart_high,
    color=band_color,
    alpha=std_alpha,
    label=r'$\mathrm{COLA\_NM4096}\ 1\sigma$',
)
ax_tl.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_tl.set_ylabel(r'$k^{3/2} P(k)$', fontsize=label_size)
ax_tl.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25)

ax_bl.semilogx(k_LCDM_Me_plot, Pk_nDGP_Me_on_LCDM / Pk_LCDM_Me_plot, lw=line_width, color=cola_color, ls='--', label=r'$\mathrm{Our\ simulation\ (mean,\ inverse\ seeds)}$')
ax_bl.semilogx(k_LCDM_Bart_plot, Pk_nDGP_over_LCDM_Bart, lw=line_width, color=bart_color, label=bart_label)
ax_bl.fill_between(
    k_LCDM_Bart_plot,
    Pk_nDGP_over_LCDM_Bart - Pk_nDGP_over_LCDM_Bart_std,
    Pk_nDGP_over_LCDM_Bart + Pk_nDGP_over_LCDM_Bart_std,
    color=band_color,
    alpha=std_alpha,
    label=r'$\mathrm{COLA\_NM4096}\ 1\sigma$',
)
ax_bl.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_bl.set_ylabel(r'$P_{\mathrm{nDGP}} / P_{\mathrm{LCDM}}$', fontsize=label_size)
ax_bl.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25)

ax_tr.semilogx(k_LCDM_Bart_plot, Pk_COLA_over_Bart, lw=line_width, color=ratio_color)
ax_tr.fill_between(k_LCDM_Bart_plot, Pk_COLA_over_Bart_low, Pk_COLA_over_Bart_high, color=ratio_color, alpha=std_alpha)
ax_tr.axhline(1.0, color=ref_color, lw=0.9, ls=':')
ax_tr.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_tr.set_ylabel(r'$P_{\mathrm{COLA}} / P_{\mathrm{COLA}}$', fontsize=label_size)

ax_br.set_visible(False)

for ax in (ax_tl, ax_tr, ax_bl):
    ax.minorticks_on()
    ax.grid(True, which='major', ls=':', color='0.82', alpha=0.75, lw=0.45)
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=tick_size, length=3.4, width=0.7)
    ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=tick_size - 0.8, length=2.0, width=0.55)
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)

plt.show()


In [ ]:
# Two-panel nDGP comparison at z=1.650 (inverse-seed mean)
k_nDGP_Bart_plot = np.asarray(k_nDGP_Bart)
Pk_nDGP_Bart_plot = np.asarray(Pk_nDGP_Bart)
Pk_nDGP_Bart_std_plot = np.asarray(Pk_nDGP_Bart_std)
k_nDGP_Me_plot = np.asarray(k_nDGP_Me_inv)
Pk_nDGP_Me_plot = np.asarray(Pk_nDGP_Me_inv)
bart_label = r'$\mathrm{COLA\_NM4096\ mean}$'

# Cut our power spectrum to the comparison's k range
mask_nDGP_Me = (k_nDGP_Me_plot >= k_nDGP_Bart_plot.min()) & (k_nDGP_Me_plot <= k_nDGP_Bart_plot.max())
k_nDGP_Me_plot = k_nDGP_Me_plot[mask_nDGP_Me]
Pk_nDGP_Me_plot = Pk_nDGP_Me_plot[mask_nDGP_Me]

Pk_nDGP_Me_on_Bart = np.interp(k_nDGP_Bart_plot, k_nDGP_Me_plot, Pk_nDGP_Me_plot)
Pk_nDGP_Bart_low = np.maximum(Pk_nDGP_Bart_plot - Pk_nDGP_Bart_std_plot, 1e-30)
Pk_nDGP_Bart_high = Pk_nDGP_Bart_plot + Pk_nDGP_Bart_std_plot
Pk_COLA_over_Bart = Pk_nDGP_Me_on_Bart / Pk_nDGP_Bart_plot
Pk_COLA_over_Bart_low = Pk_nDGP_Me_on_Bart / Pk_nDGP_Bart_high
Pk_COLA_over_Bart_high = Pk_nDGP_Me_on_Bart / Pk_nDGP_Bart_low

fig_size = (6.6, 4.35)
label_size = 10.5
tick_size = 8.8
legend_size = 8.3
line_width = 1.45
std_alpha = 0.18
cola_color = '#c47f2b'
bart_color = '#1f4e79'
ratio_color = '#2b7a78'
band_color = bart_color
ref_color = '0.35'

fig, axes = plt.subplots(2, 2, figsize=fig_size, dpi=300, constrained_layout=True)
ax_l, ax_r = axes[0]
ax_bl, ax_br = axes[1]

ax_l.semilogx(k_nDGP_Me_plot, k_nDGP_Me_plot**(3/2) * Pk_nDGP_Me_plot, lw=line_width, color=cola_color, ls='--', label=r'$\mathrm{Our\ simulation\ (mean,\ inverse\ seeds)}$')
ax_l.semilogx(k_nDGP_Bart_plot, k_nDGP_Bart_plot**(3/2) * Pk_nDGP_Bart_plot, lw=line_width, color=bart_color, label=bart_label)
ax_l.fill_between(
    k_nDGP_Bart_plot,
    k_nDGP_Bart_plot**(3/2) * (Pk_nDGP_Bart_plot - Pk_nDGP_Bart_std_plot),
    k_nDGP_Bart_plot**(3/2) * (Pk_nDGP_Bart_plot + Pk_nDGP_Bart_std_plot),
    color=band_color,
    alpha=std_alpha,
    label=r'$\mathrm{COLA\_NM4096}\ 1\sigma$',
)
ax_l.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_l.set_ylabel(r'$k^{3/2} P(k)$', fontsize=label_size)
ax_l.legend(fontsize=legend_size, frameon=False, handlelength=1.5, borderaxespad=0.25, labelspacing=0.25)

ax_r.semilogx(k_nDGP_Bart_plot, Pk_COLA_over_Bart, lw=line_width, color=ratio_color)
ax_r.fill_between(k_nDGP_Bart_plot, Pk_COLA_over_Bart_low, Pk_COLA_over_Bart_high, color=ratio_color, alpha=std_alpha)
ax_r.axhline(1.0, color=ref_color, lw=0.9, ls=':')
ax_r.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$', fontsize=label_size)
ax_r.set_ylabel(r'$P_{\mathrm{COLA}} / P_{\mathrm{COLA}}$', fontsize=label_size)

ax_bl.set_visible(False)
ax_br.set_visible(False)

for ax in (ax_l, ax_r):
    ax.minorticks_on()
    ax.grid(True, which='major', ls=':', color='0.82', alpha=0.75, lw=0.45)
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=tick_size, length=3.4, width=0.7)
    ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=tick_size - 0.8, length=2.0, width=0.55)
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)

plt.show()